<a href="https://colab.research.google.com/github/CristianPetrica/ICI_Practica_UTM/blob/Cristian-Petrica/ICI_2_Practica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 1 - Import Libraries**

In [3]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import tensorflow as tf
import random

import os
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


# **Task 2 - Import data**

In [1]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle
!kaggle datasets download -d zarif98sjs/gait-in-parkinsons-disease
!unzip gait-in-parkinsons-disease.zip -d gait_data
import os
input_dir = 'gait_data'
print(len(os.listdir(input_dir)))  # Afișează numărul de fișiere din director

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/zarif98sjs/gait-in-parkinsons-disease
License(s): unknown
Archive:  gait-in-parkinsons-disease.zip
  inflating: gait_data/GaCo01_01.txt  
  inflating: gait_data/GaCo02_01.txt  
  inflating: gait_data/GaCo02_02.txt  
  inflating: gait_data/GaCo03_01.txt  
  inflating: gait_data/GaCo03_02.txt  
  inflating: gait_data/GaCo04_01.txt  
  inflating: gait_data/GaCo04_02.txt  
  inflating: gait_data/GaCo05_01.txt  
  inflating: gait_data/GaCo05_02.txt  
  inflating: gait_data/GaCo06_01.txt  
  inflating: gait_data/GaCo06_02.txt  
  inflating: gait_data/GaCo07_01.txt  
  inflating: gait_data/GaCo07_02.txt  
  inflating: gait_data/GaCo08_01.txt  
  inflating: gait_data/GaCo08_02.txt  
  inflating: gait_data/GaCo09_01.txt  
  inflating: gait_data/GaCo09_02.txt  
  inflating: gait_data/GaCo10_01.txt  
  inflating: gait_data/GaCo10_02.txt  
  inflating: gait_data/GaCo11_01.txt  
  inflating: gait_data/GaCo12_01.txt  
  i

In [9]:

features = ['Time', 'L1' , 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8',
            'R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8',
            'Total_Force_Left', 'Total_Force_Right']

!mkdir -p CSV
!mkdir -p date  # pentru demographics

i = 0
for file in os.listdir(input_dir):
    i += 1
    if file == 'demographics.txt':
        df = pd.read_csv(os.path.join(input_dir, file), sep='\t', on_bad_lines='skip')
        name = 'date/' + file.split('.')[0] + '.csv'
        df.to_csv(name, index=False)
    elif file[:2] in ['Ga', 'Ju', 'Si']:
        df = pd.read_csv(os.path.join(input_dir, file), header=None, sep='\t', on_bad_lines='skip')
        df.columns = features
        name = 'CSV/' + file.split('.')[0] + '.csv'
        df.to_csv(name, index=False)

print(i)


309


# **Task 3 - Data Analysis**

In [39]:
import os
import pandas as pd
from scipy.signal import find_peaks

# Calea către foldere
csv_folder = 'CSV/'
demographics_path = 'date/demographics.csv'

# Încarcă datele demografice
demo_df = pd.read_csv(demographics_path)

# Inițializează lista de feature-uri
feature_list = []

for file in os.listdir(csv_folder):
    if file.endswith('.csv') and (file.startswith('Ga') or file.startswith('Ju') or file.startswith('Si')):
        file_path = os.path.join(csv_folder, file)
        df = pd.read_csv(file_path)
        patient_id = file.split('_')[0]

        # Extrage eticheta din nume
        label = 1 if 'Pt' in file else 0

        # Extrage caracteristici statistice
        df['Force_Diff'] = abs(df['Total_Force_Left'] - df['Total_Force_Right'])

        left_peaks, _ = find_peaks(df['Total_Force_Left'], height=30)
        right_peaks, _ = find_peaks(df['Total_Force_Right'], height=30)

        features = {
        'ID': patient_id,
        'mean_force_left': df['Total_Force_Left'].mean(),
        'mean_force_right': df['Total_Force_Right'].mean(),
        'std_force_left': df['Total_Force_Left'].std(),
        'std_force_right': df['Total_Force_Right'].std(),
        'step_count_left': len(left_peaks),
        'step_count_right': len(right_peaks),
        'step_symmetry_mean_abs_diff': df['Force_Diff'].mean(),  # diferență absolută
        'step_symmetry_ratio': df['Total_Force_Left'].mean() / (df['Total_Force_Right'].mean() + 1e-6),  # raport stânga/dreapta
        'label': label
        }

        # Găsește datele demografice pentru pacient
        demo_row = demo_df[demo_df['ID'] == patient_id]

        if not demo_row.empty:
            # Adaugă demograficele în feature set
            for col in ['Age', 'Gender', 'Height', 'Weight']:
                features[col] = demo_row.iloc[0][col]
        else:
            print(f"[!] Date demografice lipsă pentru {patient_id}")
            for col in ['Age', 'Gender', 'Height', 'Weight']:
                features[col] = None

        feature_list.append(features)

# Creează DataFrame final și salvează
final_df = pd.DataFrame(feature_list)
final_df.to_csv('final_dataset.csv', index=False)
print("✅ Final dataset salvat în: final_dataset.csv")


[!] Date demografice lipsă pentru SiCo25
[!] Date demografice lipsă pentru SiCo30
[!] Date demografice lipsă pentru SiCo24
[!] Date demografice lipsă pentru SiCo27
[!] Date demografice lipsă pentru SiCo29
[!] Date demografice lipsă pentru SiCo28
[!] Date demografice lipsă pentru SiCo26
✅ Final dataset salvat în: final_dataset.csv


In [40]:
import pandas as pd

# Încarcă datasetul final
df = pd.read_csv('final_dataset.csv')

# Afișează numărul de valori lipsă pe coloană
missing = df.isnull().sum()

# Afișează doar coloanele care au valori lipsă
missing_cols = missing[missing > 0]
print("Numarul total de pacienti " , len(df))
print("🔍 Coloane cu date lipsă:")
print(missing_cols)


Numarul total de pacienti  306
🔍 Coloane cu date lipsă:
Age        7
Gender     7
Height    16
Weight    12
dtype: int64


# **Task 4 - Random Forest doar pe Date de Mers**

In [41]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Încarcă datasetul
df = pd.read_csv('final_dataset.csv')

# 2. Selectează doar coloanele legate de mers
gait_features = [
    'mean_force_left', 'mean_force_right',
    'std_force_left', 'std_force_right',
    'step_count_left', 'step_count_right',
    'step_symmetry_mean_abs_diff', 'step_symmetry_ratio'
]

X = df[gait_features]
y = df['label']

# 3. Elimină rândurile care au valori lipsă DOAR în aceste coloane
X = X.dropna()
y = y[X.index]  # păstrăm aceleași etichete

# 4. Împarte în set de antrenare / testare
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Antrenează un Random Forest
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 6. Evaluează modelul
y_pred = model.predict(X_test)

print("✅ Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))


✅ Confusion Matrix:
[[ 7 14]
 [ 1 40]]

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.33      0.48        21
           1       0.74      0.98      0.84        41

    accuracy                           0.76        62
   macro avg       0.81      0.65      0.66        62
weighted avg       0.79      0.76      0.72        62



# **Task 5 - XGBoost**

In [57]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# 1. Încarcă datele
df = pd.read_csv('final_dataset.csv')

# 2. Selectează doar gait features
gait_features = [
    'mean_force_left', 'mean_force_right',
    'std_force_left', 'std_force_right',
    'step_count_left', 'step_count_right',
    'step_symmetry_mean_abs_diff'
]
X = df[gait_features].dropna()
y = df.loc[X.index, 'label']

# 3. Aplica SMOTE pentru echilibrare
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# 4. Împarte în train/test
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# 5. Creează și antrenează modelul XGBoost
model_xgb = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
model_xgb.fit(X_train, y_train)

# 6. Evaluare
y_pred = model_xgb.predict(X_test)

print("🔷 XGBoost SMOTE - Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\n🔷 XGBoost SMOTE - Classification Report:")
print(classification_report(y_test, y_pred))


🔷 XGBoost SMOTE - Confusion Matrix:
[[34  2]
 [ 7 43]]

🔷 XGBoost SMOTE - Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.94      0.88        36
           1       0.96      0.86      0.91        50

    accuracy                           0.90        86
   macro avg       0.89      0.90      0.89        86
weighted avg       0.90      0.90      0.90        86



# **Task 6 - SVM**

In [49]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

# 1. Încarcă datele
df = pd.read_csv('final_dataset.csv')

# 2. Selectează doar gait features
gait_features = [
    'mean_force_left', 'mean_force_right',
    'std_force_left', 'std_force_right',
    'step_count_left', 'step_count_right',
    'step_symmetry_mean_abs_diff' , 'step_symmetry_ratio'
]
X = df[gait_features].dropna()
y = df.loc[X.index, 'label']

# 3. Împarte în train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Creează modelul SVM cu ponderi balansate
model_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
)
model_svm.fit(X_train, y_train)

# 5. Prezicere și evaluare
y_pred = model_svm.predict(X_test)

print("🔶 SVM Balanced - Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\n🔶 SVM Balanced - Classification Report:")
print(classification_report(y_test, y_pred))


🔶 SVM Balanced - Confusion Matrix:
[[11 10]
 [ 9 32]]

🔶 SVM Balanced - Classification Report:
              precision    recall  f1-score   support

           0       0.55      0.52      0.54        21
           1       0.76      0.78      0.77        41

    accuracy                           0.69        62
   macro avg       0.66      0.65      0.65        62
weighted avg       0.69      0.69      0.69        62



# **Task 7 - KNN**

In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Încarcă datele
df = pd.read_csv('final_dataset.csv')

# 2. Feature-uri de bază biomecanice
gait_features = [
    'mean_force_left', 'mean_force_right',
    'std_force_left', 'std_force_right',
    'step_count_left', 'step_count_right',
    'step_symmetry_mean_abs_diff' , 'step_symmetry_ratio'
]

X = df[gait_features]
y = df['label']

# 3. Imputare valori lipsă
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=gait_features)

# 4. Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# 5. Creează pipeline KNN (cu standardizare)
model_knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=7)  # poți testa și cu 3, 7, etc.
)
model_knn.fit(X_train, y_train)

# 6. Evaluare
y_pred = model_knn.predict(X_test)

print("🔵 KNN - Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\n🔵 KNN - Classification Report:")
print(classification_report(y_test, y_pred))


🔵 KNN - Confusion Matrix:
[[10 11]
 [ 3 38]]

🔵 KNN - Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.48      0.59        21
           1       0.78      0.93      0.84        41

    accuracy                           0.77        62
   macro avg       0.77      0.70      0.72        62
weighted avg       0.77      0.77      0.76        62

